# FiQA BM25 + Compressed Dense RRF Benchmark

## Goal
Evaluate whether BM25 can recover relevant documents missed by compressed dense ANN retrieval.

### Compared methods
1. BM25 only
2. Float32 exact dense retrieval
3. IVF-PQ ADC, M=96, nprobe=16
4. Native Faiss OPQMatrix + IVF-PQ ADC, M=96, nprobe=16
5. BM25 + IVF-PQ RRF
6. BM25 + OPQ-IVF-PQ RRF

### Evaluation discipline
- FiQA queries are deterministically split into calibration (20%) and held-out (80%).
- This first run uses a fixed RRF configuration: `rrf_k=60`, equal weights.
- The held-out split is the primary result for later claims.
- No RRF tuning is performed in this notebook.


In [ ]:
# 1. Install dependencies
%pip uninstall -y faiss-cpu faiss-gpu faiss-gpu-cu11 faiss-gpu-cu12
%pip install -q --upgrade --upgrade-strategy only-if-needed \
  "transformers==4.49.0" \
  "sentence-transformers==3.4.1" \
  "tokenizers>=0.21,<0.22" \
  "sentencepiece>=0.2.0" \
  "safetensors>=0.4.5" \
  "faiss-gpu-cu12" \
  "rank-bm25==0.2.2" \
  "pandas>=2.0" \
  "matplotlib>=3.7"

print("Install complete. Restart the Colab runtime once if Faiss GPU is unavailable.")

In [ ]:
# 2. Imports, configuration, and reproducibility
import csv
import gc
import hashlib
import json
import random
import re
import time
import urllib.request
import zipfile
from pathlib import Path

import faiss
import faiss.contrib.torch_utils
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. In Colab, select a T4 GPU runtime.")

DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

if faiss.get_num_gpus() < 1:
    raise RuntimeError(
        "Faiss GPU backend was not detected. Restart the runtime, then rerun from Cell 1."
    )

FIQA_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
DATA_DIR = Path("beir_data")
FIQA_DIR = DATA_DIR / "fiqa"
RESULT_DIR = Path("results/hybrid_bm25_rrf_fiqa")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
DOC_EMBED_BATCH_SIZE = 256
QUERY_EMBED_BATCH_SIZE = 256

CANDIDATE_K = 100
FINAL_K = 10

FAISS_NLIST = 256
FAISS_NPROBE = 16
PQ_M = 96
PQ_NBITS = 8
FAISS_TRAIN_POINTS = 24_000
FAISS_GPU_BATCH_SIZE = 64

RRF_K = 60
RRF_W_SPARSE = 1.0
RRF_W_DENSE = 1.0

CALIBRATION_MODULUS = 5   # deterministic 20% calibration split
CALIBRATION_BUCKET = 0

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Faiss:", faiss.__version__)
print("Faiss visible GPUs:", faiss.get_num_gpus())

In [ ]:
# 3. Load FiQA and create deterministic calibration / held-out split

def download_and_extract_fiqa():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / "fiqa.zip"

    if not FIQA_DIR.exists():
        if not zip_path.exists():
            print("Downloading FiQA...")
            urllib.request.urlretrieve(FIQA_URL, zip_path)

        print("Extracting FiQA...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DATA_DIR)

    required = [
        FIQA_DIR / "corpus.jsonl",
        FIQA_DIR / "queries.jsonl",
        FIQA_DIR / "qrels" / "test.tsv",
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"FiQA extraction incomplete: {missing}")


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def read_qrels(path):
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            qid = str(row["query-id"])
            docid = str(row["corpus-id"])
            score = int(row["score"])
            result.setdefault(qid, {})[docid] = score
    return result


def split_name(qid):
    bucket = int(hashlib.sha256(qid.encode("utf-8")).hexdigest(), 16) % CALIBRATION_MODULUS
    return "calibration" if bucket == CALIBRATION_BUCKET else "heldout"


download_and_extract_fiqa()

corpus_rows = read_jsonl(FIQA_DIR / "corpus.jsonl")
query_rows = read_jsonl(FIQA_DIR / "queries.jsonl")
all_qrels = read_qrels(FIQA_DIR / "qrels" / "test.tsv")

doc_ids = [str(row["_id"]) for row in corpus_rows]
doc_texts = [
    ((row.get("title") or "") + "\n" + (row.get("text") or "")).strip()
    for row in corpus_rows
]
doc_id_to_index = {doc_id: i for i, doc_id in enumerate(doc_ids)}

query_ids, query_texts, qrels = [], [], {}
for row in query_rows:
    qid = str(row["_id"])
    if qid not in all_qrels:
        continue

    relevant = {
        doc_id: score
        for doc_id, score in all_qrels[qid].items()
        if doc_id in doc_id_to_index and score > 0
    }
    if relevant:
        query_ids.append(qid)
        query_texts.append(str(row["text"]))
        qrels[qid] = relevant

query_splits = np.array([split_name(qid) for qid in query_ids])
calibration_mask = query_splits == "calibration"
heldout_mask = query_splits == "heldout"

print(f"Documents: {len(doc_ids):,}")
print(f"Evaluation queries: {len(query_ids):,}")
print(f"Calibration queries: {int(calibration_mask.sum()):,}")
print(f"Held-out queries: {int(heldout_mask.sum()):,}")

In [ ]:
# 4. BM25, embedding generation, Faiss retrieval, RRF, and evaluation helpers

TOKEN_PATTERN = re.compile(r"[A-Za-z0-9]+")

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


def encode_texts_gpu(texts, model_name, batch_size):
    model = SentenceTransformer(model_name, device="cuda")
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device="cuda",
    )
    del model
    torch.cuda.empty_cache()
    return torch.from_numpy(vectors.astype(np.float32)).to(DEVICE)


def per_query_metrics(rankings, query_ids, doc_ids, qrels, k=10):
    recall = np.zeros(len(query_ids), dtype=np.float64)
    mrr = np.zeros(len(query_ids), dtype=np.float64)
    ndcg = np.zeros(len(query_ids), dtype=np.float64)

    discounts = 1.0 / np.log2(np.arange(2, k + 2))

    for i, (row, qid) in enumerate(zip(rankings[:, :k], query_ids)):
        relevance_map = qrels[qid]
        relevant = set(relevance_map)

        retrieved_ids = [
            doc_ids[int(idx)] if int(idx) >= 0 else None
            for idx in row
        ]
        retrieved_set = {docid for docid in retrieved_ids if docid is not None}
        recall[i] = len(relevant & retrieved_set) / len(relevant)

        gains = np.array(
            [relevance_map.get(docid, 0) if docid is not None else 0 for docid in retrieved_ids],
            dtype=np.float64,
        )

        first_rel = np.where(gains > 0)[0]
        mrr[i] = 1.0 / (first_rel[0] + 1) if len(first_rel) else 0.0

        dcg = np.sum((2**gains - 1) * discounts)
        ideal = np.sort(np.asarray(list(relevance_map.values()), dtype=np.float64))[::-1][:k]
        idcg = np.sum((2**ideal - 1) * discounts[:len(ideal)])
        ndcg[i] = dcg / idcg if idcg > 0 else 0.0

    return {
        "recall_at_10": recall,
        "mrr_at_10": mrr,
        "ndcg_at_10": ndcg,
    }


def aggregate_metrics(per_query, mask):
    return {
        name: float(values[mask].mean())
        for name, values in per_query.items()
    }


def bm25_search(bm25, texts, k):
    rankings = []
    start = time.perf_counter()

    for text in texts:
        scores = bm25.get_scores(tokenize(text))
        candidate = np.argpartition(scores, -k)[-k:]
        candidate = candidate[np.argsort(scores[candidate])[::-1]]
        rankings.append(candidate.astype(np.int64))

    elapsed = time.perf_counter() - start
    return np.vstack(rankings), elapsed


def gpu_search(index, queries_gpu, k, batch_size=FAISS_GPU_BATCH_SIZE):
    for _ in range(3):
        index.search(queries_gpu[:min(batch_size, len(queries_gpu))].contiguous(), k)

    torch.cuda.synchronize()
    rankings = []
    batch_per_query_ms = []
    timed_queries = 0
    timed_seconds = 0.0

    for start in range(0, len(queries_gpu), batch_size):
        batch = queries_gpu[start:start + batch_size].contiguous()

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _, ids = index.search(batch, k)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        if isinstance(ids, torch.Tensor):
            ids = ids.detach().cpu().numpy()
        else:
            ids = np.asarray(ids)

        rankings.append(ids)

        if len(batch) == batch_size:
            timed_seconds += elapsed
            timed_queries += len(batch)
            batch_per_query_ms.append(elapsed * 1000.0 / len(batch))

    return (
        np.concatenate(rankings, axis=0),
        {
            "search_seconds": float(timed_seconds),
            "timed_query_count": int(timed_queries),
            "p50_latency_ms": float(np.median(batch_per_query_ms)),
            "p95_latency_ms": float(np.percentile(batch_per_query_ms, 95)),
            "queries_per_second": float(timed_queries / timed_seconds),
        },
    )


def rrf_fuse(sparse_rankings, dense_rankings, rrf_k=RRF_K,
             w_sparse=RRF_W_SPARSE, w_dense=RRF_W_DENSE, final_k=FINAL_K):
    fused = np.empty((len(sparse_rankings), final_k), dtype=np.int64)

    for i, (sparse_row, dense_row) in enumerate(zip(sparse_rankings, dense_rankings)):
        scores = {}

        for rank, doc_idx in enumerate(sparse_row, start=1):
            doc_idx = int(doc_idx)
            scores[doc_idx] = scores.get(doc_idx, 0.0) + w_sparse / (rrf_k + rank)

        for rank, doc_idx in enumerate(dense_row, start=1):
            doc_idx = int(doc_idx)
            scores[doc_idx] = scores.get(doc_idx, 0.0) + w_dense / (rrf_k + rank)

        ordered = sorted(scores.items(), key=lambda item: (-item[1], item[0]))
        fused[i] = [doc_idx for doc_idx, _ in ordered[:final_k]]

    return fused


def mean_candidate_overlap(left, right):
    return float(np.mean([
        len(set(a.tolist()) & set(b.tolist())) / CANDIDATE_K
        for a, b in zip(left, right)
    ]))


def win_loss_tie(reference_ndcg, hybrid_ndcg, eps=1e-12):
    delta = hybrid_ndcg - reference_ndcg
    return {
        "wins": int(np.sum(delta > eps)),
        "losses": int(np.sum(delta < -eps)),
        "ties": int(np.sum(np.abs(delta) <= eps)),
        "mean_delta_ndcg_at_10": float(np.mean(delta)),
    }

In [ ]:
# 5. Build indexes and run the complete benchmark

print("Building BM25 index...")
bm25_build_start = time.perf_counter()
bm25 = BM25Okapi([tokenize(text) for text in doc_texts])
bm25_build_seconds = time.perf_counter() - bm25_build_start

print("Encoding documents...")
X_docs = encode_texts_gpu(doc_texts, EMBEDDING_MODEL, DOC_EMBED_BATCH_SIZE)

print("Encoding queries...")
X_queries = encode_texts_gpu(query_texts, EMBEDDING_MODEL, QUERY_EMBED_BATCH_SIZE)

N_DOCS, D = X_docs.shape
assert D % PQ_M == 0, f"Embedding dimension {D} must be divisible by M={PQ_M}"

X_docs_gpu = X_docs.contiguous().float()
X_queries_gpu = X_queries.contiguous().float()

train_count = min(FAISS_TRAIN_POINTS, N_DOCS)
train_indices = torch.randperm(N_DOCS, device=DEVICE)[:train_count]
X_train_gpu = X_docs_gpu[train_indices].contiguous()

gpu_resources = faiss.StandardGpuResources()
gpu_resources.setDefaultNullStreamAllDevices()
gpu_resources.setTempMemory(768 * 1024 * 1024)

flat_config = faiss.GpuIndexFlatConfig()
flat_config.device = 0
flat_config.useFloat16 = False

ivfpq_config = faiss.GpuIndexIVFPQConfig()
ivfpq_config.device = 0
ivfpq_config.useFloat16LookupTables = True
ivfpq_config.indicesOptions = faiss.INDICES_32_BIT
ivfpq_config.interleavedLayout = True

print("Running BM25...")
bm25_rankings, bm25_latency = bm25_search(bm25, query_texts, CANDIDATE_K)

print("Building exact Float32 GPU baseline...")
gpu_flat = faiss.GpuIndexFlatIP(gpu_resources, D, flat_config)
gpu_flat.add(X_docs_gpu)
flat_rankings, flat_latency = gpu_search(gpu_flat, X_queries_gpu, CANDIDATE_K)

print("Training IVF-PQ M=96...")
pq_index = faiss.GpuIndexIVFPQ(
    gpu_resources, D, FAISS_NLIST, PQ_M, PQ_NBITS,
    faiss.METRIC_INNER_PRODUCT, ivfpq_config
)
pq_index.train(X_train_gpu)
pq_index.add(X_docs_gpu)
pq_index.nprobe = FAISS_NPROBE
pq_rankings, pq_latency = gpu_search(pq_index, X_queries_gpu, CANDIDATE_K)

print("Training native Faiss OPQMatrix + IVF-PQ M=96...")
train_np = np.ascontiguousarray(X_train_gpu.detach().cpu().numpy().astype(np.float32))
docs_np = np.ascontiguousarray(X_docs_gpu.detach().cpu().numpy().astype(np.float32))
queries_np = np.ascontiguousarray(X_queries_gpu.detach().cpu().numpy().astype(np.float32))

native_opq = faiss.OPQMatrix(D, PQ_M)
native_opq.niter = 25
native_opq.niter_pq = 4
native_opq.train(train_np)

docs_rot_np = np.ascontiguousarray(native_opq.apply_py(docs_np).astype(np.float32))
queries_rot_np = np.ascontiguousarray(native_opq.apply_py(queries_np).astype(np.float32))
faiss.normalize_L2(docs_rot_np)
faiss.normalize_L2(queries_rot_np)

docs_rot_gpu = torch.from_numpy(docs_rot_np).to(DEVICE).contiguous()
queries_rot_gpu = torch.from_numpy(queries_rot_np).to(DEVICE).contiguous()
opq_train_gpu = docs_rot_gpu[train_indices].contiguous()

opq_index = faiss.GpuIndexIVFPQ(
    gpu_resources, D, FAISS_NLIST, PQ_M, PQ_NBITS,
    faiss.METRIC_INNER_PRODUCT, ivfpq_config
)
opq_index.train(opq_train_gpu)
opq_index.add(docs_rot_gpu)
opq_index.nprobe = FAISS_NPROBE
opq_rankings, opq_latency = gpu_search(opq_index, queries_rot_gpu, CANDIDATE_K)

fusion_start = time.perf_counter()
bm25_pq_rrf_rankings = rrf_fuse(bm25_rankings, pq_rankings)
pq_rrf_fusion_seconds = time.perf_counter() - fusion_start

fusion_start = time.perf_counter()
bm25_opq_rrf_rankings = rrf_fuse(bm25_rankings, opq_rankings)
opq_rrf_fusion_seconds = time.perf_counter() - fusion_start

methods = {
    "bm25": {"rankings": bm25_rankings, "latency": bm25_latency},
    "float32_flat_ip": {"rankings": flat_rankings, "latency": flat_latency},
    "ivfpq_m96_np16": {"rankings": pq_rankings, "latency": pq_latency},
    "opq_ivfpq_m96_np16": {"rankings": opq_rankings, "latency": opq_latency},
    "bm25_ivfpq_rrf": {
        "rankings": bm25_pq_rrf_rankings,
        "latency": {
            "search_seconds": float(
                bm25_latency + pq_latency["search_seconds"] + pq_rrf_fusion_seconds
            ),
            "component_latency": "sequential_bm25_plus_ivfpq_plus_rrf",
        },
    },
    "bm25_opq_ivfpq_rrf": {
        "rankings": bm25_opq_rrf_rankings,
        "latency": {
            "search_seconds": float(
                bm25_latency + opq_latency["search_seconds"] + opq_rrf_fusion_seconds
            ),
            "component_latency": "sequential_bm25_plus_opq_ivfpq_plus_rrf",
        },
    },
}

summary_rows = []
per_query_frames = []

for method, payload in methods.items():
    rankings = payload["rankings"]
    scores = per_query_metrics(rankings, query_ids, doc_ids, qrels, FINAL_K)

    for split, mask in {
        "all": np.ones(len(query_ids), dtype=bool),
        "calibration": calibration_mask,
        "heldout": heldout_mask,
    }.items():
        row = {
            "method": method,
            "split": split,
            "query_count": int(mask.sum()),
            **aggregate_metrics(scores, mask),
        }

        if method == "bm25":
            row.update({
                "candidate_k": CANDIDATE_K,
                "bm25_build_seconds": float(bm25_build_seconds),
                "sparse_search_seconds_all_queries": float(bm25_latency),
                "sparse_latency_per_query_ms_all_queries": float(bm25_latency * 1000 / len(query_ids)),
            })
        elif method in {"float32_flat_ip", "ivfpq_m96_np16", "opq_ivfpq_m96_np16"}:
            row.update(payload["latency"])
        else:
            row.update(payload["latency"])

        summary_rows.append(row)

    frame = pd.DataFrame({
        "query_id": query_ids,
        "split": query_splits,
        "method": method,
        "recall_at_10": scores["recall_at_10"],
        "mrr_at_10": scores["mrr_at_10"],
        "ndcg_at_10": scores["ndcg_at_10"],
    })
    per_query_frames.append(frame)

summary_df = pd.DataFrame(summary_rows)
per_query_df = pd.concat(per_query_frames, ignore_index=True)

analysis_rows = []
for dense_method, hybrid_method, dense_rankings, hybrid_rankings in [
    ("ivfpq_m96_np16", "bm25_ivfpq_rrf", pq_rankings, bm25_pq_rrf_rankings),
    ("opq_ivfpq_m96_np16", "bm25_opq_ivfpq_rrf", opq_rankings, bm25_opq_rrf_rankings),
]:
    # Each method is appended in the same original query order.
    # Keep that order so calibration_mask / heldout_mask remain aligned.
    dense_ndcg = per_query_df[
        per_query_df["method"] == dense_method
    ]["ndcg_at_10"].to_numpy()

    hybrid_ndcg = per_query_df[
        per_query_df["method"] == hybrid_method
    ]["ndcg_at_10"].to_numpy()

    for split, mask in {
        "all": np.ones(len(query_ids), dtype=bool),
        "calibration": calibration_mask,
        "heldout": heldout_mask,
    }.items():
        stats = win_loss_tie(dense_ndcg[mask], hybrid_ndcg[mask])
        analysis_rows.append({
            "dense_method": dense_method,
            "hybrid_method": hybrid_method,
            "split": split,
            "query_count": int(mask.sum()),
            "bm25_dense_candidate_overlap_at_100": mean_candidate_overlap(
                bm25_rankings[mask],
                dense_rankings[mask],
            ),
            **stats,
        })

analysis_df = pd.DataFrame(analysis_rows)

summary_df.to_csv(RESULT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
per_query_df.to_csv(RESULT_DIR / "per_query_metrics.csv", index=False, encoding="utf-8-sig")
analysis_df.to_csv(RESULT_DIR / "hybrid_analysis.csv", index=False, encoding="utf-8-sig")

metadata = {
    "dataset": "FiQA-2018 / BEIR",
    "embedding_model": EMBEDDING_MODEL,
    "corpus_documents": int(N_DOCS),
    "evaluation_queries": int(len(query_ids)),
    "candidate_k": CANDIDATE_K,
    "final_k": FINAL_K,
    "rrf": {
        "rrf_k": RRF_K,
        "w_sparse": RRF_W_SPARSE,
        "w_dense": RRF_W_DENSE,
        "tuned": False,
    },
    "split": {
        "calibration_rule": "sha256(query_id) mod 5 == 0",
        "calibration_queries": int(calibration_mask.sum()),
        "heldout_queries": int(heldout_mask.sum()),
    },
    "faiss": {
        "nlist": FAISS_NLIST,
        "nprobe": FAISS_NPROBE,
        "m": PQ_M,
        "nbits": PQ_NBITS,
        "native_opq": True,
    },
    "timing_note": (
        "BM25 is measured as sequential CPU scoring over the full corpus. "
        "Dense timing is synchronized GPU search timing. Hybrid sequential time "
        "combines sparse retrieval, dense retrieval, and CPU-side RRF fusion; "
        "it is not an end-to-end API latency claim."
    ),
}
(RESULT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

heldout_summary = (
    summary_df[summary_df["split"] == "heldout"]
    [["method", "query_count", "recall_at_10", "mrr_at_10", "ndcg_at_10"]]
    .sort_values("ndcg_at_10", ascending=False)
)

print("\nHeld-out quality results")
display(heldout_summary)

print("\nHybrid overlap and win/loss/tie analysis")
display(analysis_df[analysis_df["split"] == "heldout"])

print("\nSaved outputs:")
for path in sorted(RESULT_DIR.iterdir()):
    print("-", path)

del train_np, docs_np, queries_np, docs_rot_np, queries_rot_np
gc.collect()
torch.cuda.empty_cache()